# Lab 6: Deep Learning for NLP

In [1]:
import re

import numpy as np
import pandas as pd
import nltk
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

try:
    import gensim
except ImportError:
    %pip install -q gensim
    import gensim
from gensim.models import Word2Vec

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

np.random.seed(42)

## Load, preprocess and split the data (as in the lab)

In [2]:
df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
print(df["label"].value_counts())
df.head()

Number of reviews: 1000
label
1    500
0    500
Name: count, dtype: int64


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["review"] = df["review"].apply(preprocess_text)

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## Task 1: Represent reviews using Word2Vec

In [ ]:
# tokenize the training and testing reviews
train_tokens = [word_tokenize(review) for review in X_train]
test_tokens = [word_tokenize(review) for review in X_test]

# Word2Vec trained on the training reviews only
w2v_model = Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1,
                     epochs=100, seed=42, workers=1)

print("Vocabulary size:", len(w2v_model.wv))
print("Vector size    :", w2v_model.wv.vector_size)

Vocabulary size: 1786
Vector size    : 200


In [ ]:
# (words that are not in the training vocabulary are skipped)
def review_to_vector(tokens):
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v_model.wv.vector_size)

X_train_w2v = np.array([review_to_vector(tokens) for tokens in train_tokens])
X_test_w2v = np.array([review_to_vector(tokens) for tokens in test_tokens])

# convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

print("X_train_tensor:", X_train_tensor.shape)
print("X_test_tensor :", X_test_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

X_train_tensor: torch.Size([800, 200])
X_test_tensor : torch.Size([200, 200])
y_train_tensor: torch.Size([800, 1])


## Task 2: Build a different neural network

In [6]:
torch.manual_seed(42)
input_size = X_train_tensor.shape[1]     # 200

model = nn.Sequential(
    nn.Linear(input_size, 128),   # input -> hidden layer 1
    nn.ReLU(),

    nn.Linear(128, 64),           # hidden layer 2
    nn.ReLU(),

    nn.Linear(64, 32),            # hidden layer 3
    nn.ReLU(),

    nn.Linear(32, 1)              # output neuron
)

print(model)

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [7]:
# mini-batches of 32 reviews (one epoch = one pass over the whole training set)
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)

epochs = 15

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)                    # 1. forward pass
        loss = loss_function(outputs, y_batch)      # 2. loss
        loss.backward()                             # 3. backpropagation
        optimizer.step()                            # 4. update weights
        epoch_loss += loss.item() * len(X_batch)

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss / len(X_train_tensor):.4f}")

Epoch 1/15 - Loss: 0.6887
Epoch 2/15 - Loss: 0.6414
Epoch 3/15 - Loss: 0.5737
Epoch 4/15 - Loss: 0.5248
Epoch 5/15 - Loss: 0.4953
Epoch 6/15 - Loss: 0.4700
Epoch 7/15 - Loss: 0.4296


Epoch 8/15 - Loss: 0.4203
Epoch 9/15 - Loss: 0.3810
Epoch 10/15 - Loss: 0.3433
Epoch 11/15 - Loss: 0.3225
Epoch 12/15 - Loss: 0.3111
Epoch 13/15 - Loss: 0.2502


Epoch 14/15 - Loss: 0.2221
Epoch 15/15 - Loss: 0.1949


In [8]:
model.eval()

with torch.no_grad():
    test_probabilities = torch.sigmoid(model(X_test_tensor))
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)
print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.755
